# Graph RAG retriever (Neo4j + embeddings)

Connects to the same vector index as `store.py` / `retrieve.py` (`Chunk` + `chunk_embeddings`).

**Prerequisites:** `uv run python src/store.py` has been run, `.env` has Neo4j + Azure OpenAI vars.

Run this notebook with the project root as the current working directory (e.g. open the folder in Cursor and set the kernel cwd to `graph_rag` if needed).

In [1]:
import os
from dataclasses import dataclass

import pandas as pd
from dotenv import load_dotenv
from IPython.display import display
from langchain_community.vectorstores import Neo4jVector
from langchain_openai import AzureOpenAIEmbeddings
from langchain_core.documents import Document
from neo4j import GraphDatabase

load_dotenv()

True

In [2]:
NEO4J_URI = os.environ["NEO4J_URI"]
NEO4J_USERNAME = os.environ["NEO4J_USERNAME"]
NEO4J_PASSWORD = os.environ["NEO4J_PASSWORD"]

INDEX_NAME = "chunk_embeddings"
NODE_LABEL = "Chunk"


def make_embeddings() -> AzureOpenAIEmbeddings:
    return AzureOpenAIEmbeddings(
        azure_deployment=os.environ["AZURE_EMBEDDING_DEPLOYMENT"].strip(),
        azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"].strip(),
        api_key=os.environ["AZURE_OPENAI_API_KEY"].strip(),
        api_version=os.environ["AZURE_OPENAI_API_VERSION"].strip(),
    )


def test_embeddings() -> None:
    """Call Azure embeddings once to validate env vars/deployment."""
    emb = make_embeddings()
    v = emb.embed_query("ping")
    print(f"Embeddings OK. dim={len(v)}")


def get_vector_store() -> Neo4jVector:
    # NOTE: Neo4jVector calculates embedding dimension immediately, which triggers an
    # embeddings API call. If you get a 404 here, fix AZURE_* env vars / deployment.
    embeddings = make_embeddings()
    return Neo4jVector.from_existing_index(
        embedding=embeddings,
        url=NEO4J_URI,
        username=NEO4J_USERNAME,
        password=NEO4J_PASSWORD,
        index_name=INDEX_NAME,
        node_label=NODE_LABEL,
        text_node_property="page_content",
        embedding_node_property="embedding",
    )


def document_to_row(doc: Document, score: float | None = None) -> dict:
    m = doc.metadata or {}
    row = {
        "section": m.get("section", ""),
        "filing_type": m.get("filing_type", ""),
        "period": m.get("period", ""),
        "page": m.get("page_number", ""),
        "chunk_type": m.get("chunk_type", ""),
        "chunk_index": m.get("chunk_index", ""),
        "preview": (doc.page_content or "")[:400].replace("\n", " "),
    }
    if score is not None:
        row["relevance_score"] = score
    return row


def retrieve_chunks(
    question: str,
    *,
    k: int = 5,
    filter_dict: dict | None = None,
    with_scores: bool = False,
) -> list[Document] | list[tuple[Document, float]]:
    vs = get_vector_store()
    if with_scores:
        return vs.similarity_search_with_relevance_scores(question, k=k, filter=filter_dict)
    kwargs = {"k": k}
    if filter_dict:
        kwargs["filter"] = filter_dict
    return vs.similarity_search(question, **kwargs)


def chunks_to_dataframe(
    results: list[Document] | list[tuple[Document, float]],
) -> pd.DataFrame:
    if not results:
        return pd.DataFrame()
    if isinstance(results[0], tuple):
        rows = [document_to_row(d, s) for d, s in results]  # type: ignore[union-attr]
    else:
        rows = [document_to_row(d) for d in results]  # type: ignore[union-attr]
    return pd.DataFrame(rows)


## 0) First: validate embeddings

If you see a 404/NotFoundError, it usually means one of these is wrong:
- `AZURE_OPENAI_ENDPOINT`
- `AZURE_EMBEDDING_DEPLOYMENT`
- `AZURE_OPENAI_API_VERSION`
- your Azure resource / deployment was deleted or renamed

Run the next cell once. If it passes, proceed to retrieval.

In [3]:
# Validate Azure embeddings configuration
try:
    test_embeddings()
except Exception as e:
    print("Embeddings test failed.")
    raise

# If embeddings work, run semantic retrieval
K = 5
QUESTION = "What are MSCI's main business segments?"

out = retrieve_chunks(QUESTION, k=K, with_scores=True)
df = chunks_to_dataframe(out)
print(QUESTION)
display(df)


NotFoundError: Error code: 404 - {'error': {'code': '404', 'message': 'Resource not found'}}

## Filtered search (metadata)
Same as `retrieve.py` — pre-filter `Chunk` metadata before vector match.

In [4]:
QUESTION2 = "What revenue risks does the Index segment face?"
fltr = {
    "filing_type": "10-K",
    "section": "Item 1A - Risk Factors",
}

out2 = retrieve_chunks(QUESTION2, k=5, filter_dict=fltr, with_scores=True)
print(QUESTION2, fltr)
display(chunks_to_dataframe(out2))


NotFoundError: Error code: 404 - {'error': {'code': '404', 'message': 'Resource not found'}}

## Full text for one hit
Inspect a single chunk without truncation in the table above.

In [ ]:
if out:
    doc0 = out[0][0] if isinstance(out[0], tuple) else out[0]
    print(doc0.page_content)


## (Optional) Cypher: 13F `Manager` → MSCI
Uses the graph from `load_13f_graph.py` — not the vector index. Example: filers with “New York” in the business address.

In [ ]:
CYPHER_13F_NY = """
MATCH (m:Manager)-[r:OWNS_STOCK_IN]->(c:Company {ticker: "MSCI"})
WHERE toUpper(m.address) CONTAINS "NEW YORK, NY"
RETURN m.cik AS cik, m.name AS name, m.address AS address, r.value AS value_usd, r.shares AS shares, r.as_of AS as_of
ORDER BY r.value DESC
LIMIT 50
"""

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))
with driver.session() as session:
    recs = [dict(r) for r in session.run(CYPHER_13F_NY)]
driver.close()
display(pd.DataFrame(recs))
